# Servir `fireball-qwen3-4b-lora-10k` con vLLM + ngrok

Este notebook levanta un servidor OpenAI-compatible con **vLLM** que sirve el adapter LoRA
[`fireball-nlp/fireball-qwen3-4b-lora-10k`](https://huggingface.co/fireball-nlp/fireball-qwen3-4b-lora-10k)
(base `Qwen/Qwen3-4B-Instruct-2507`), y lo expone a internet con un túnel de **ngrok** para que
el juego NLP-RPG (hosteado en GitHub Pages) pueda usarlo.

Cada vez que quieras jugar: corré este notebook de arriba a abajo, copiá la URL que se imprime al
final, y pegala en el panel de ajustes (⚙ LINK) de la app. Cuando termines de jugar, parar/cerrar
la sesión de Kaggle apaga el servidor.

**Requisitos antes de correr:**
- En *Notebook Settings* → *Accelerator*: elegir GPU (P100 o T4 x2).
- En *Notebook Settings*: habilitar **Internet**.
- Tener una cuenta gratuita de [ngrok](https://ngrok.com) y su authtoken.

## Kaggle Secrets necesarios

Antes de correr, ir a **Add-ons → Secrets** en este notebook y agregar:

- `NGROK_AUTHTOKEN` (**obligatorio**) — desde https://dashboard.ngrok.com/get-started/your-authtoken
- `VLLM_API_KEY` (opcional, recomendado) — inventá cualquier string; va a ser la "API Key" que
  peguás en el panel de ajustes de la app. Sin esto, cualquiera con la URL podría usar tu servidor.
- `NGROK_DOMAIN` (opcional) — solo si reservaste un dominio fijo gratis en tu cuenta de ngrok.
  Si lo dejás sin configurar, ngrok genera una URL nueva (al azar) cada vez que corras esto — es
  lo esperado, y por eso la app tiene un panel de ajustes en vez de una URL fija en el código.

In [ ]:
# Celda 1 — instalar dependencias
# Nota: Kaggle ya trae un torch con CUDA; si en el futuro esta celda falla por
# incompatibilidad de versiones, piné una versión concreta de vllm (ej. vllm==X.Y.Z)
# una vez que hayas confirmado que funciona.
!pip install -q vllm pyngrok

In [ ]:
# Celda 2 — leer secrets de Kaggle
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
NGROK_AUTHTOKEN = secrets.get_secret("NGROK_AUTHTOKEN")

try:
    VLLM_API_KEY = secrets.get_secret("VLLM_API_KEY")
except Exception:
    VLLM_API_KEY = None

try:
    NGROK_DOMAIN = secrets.get_secret("NGROK_DOMAIN")
except Exception:
    NGROK_DOMAIN = None

print("API key configurada:", bool(VLLM_API_KEY))
print("Dominio fijo configurado:", bool(NGROK_DOMAIN))

## ⚠️ Advertencia: rank del LoRA (`--max-lora-rank`)

La model card de `fireball-nlp/fireball-qwen3-4b-lora-10k` **no especifica el rank** (`r`) del
adapter. Antes de correr la celda de `vllm serve`:

1. Abrí https://huggingface.co/fireball-nlp/fireball-qwen3-4b-lora-10k/blob/main/adapter_config.json
2. Mirá el valor de `"r"`.
3. vLLM solo acepta `--max-lora-rank` de este conjunto: `{1, 8, 16, 32, 64, 128, 256, 320, 512}`
   (default 16). Si `r` no es exactamente uno de esos valores, **redondeá hacia arriba** al
   siguiente permitido (ej. si `r=24`, usá `32`) — no pongas el valor literal del adapter.
4. Ajustá `MAX_LORA_RANK` en la celda siguiente si hace falta.

In [ ]:
MAX_LORA_RANK = 16  # !! Revisá adapter_config.json ("r") en HF y subi este valor si corresponde

## (Opcional pero recomendado) Chequear flags de vLLM antes de lanzar el servidor

La CLI de vLLM cambia entre versiones. Antes de confiar en los flags de la celda siguiente,
corré esto para confirmar que existen con esos nombres en la versión instalada:
```
!vllm serve --help | grep -E "lora-modules|max-lora-rank|allowed-origins|api-key|enable-lora"
```

In [ ]:
# Celda 3 — lanzar vllm serve en background
import subprocess

# Este alias DEBE coincidir con MODEL_NAME en src/llmService.ts
LORA_ALIAS = "fireball-qwen3-4b-lora-10k"
PORT = 8000

# Qwen3-4B-Instruct-2507 trae max_model_len=262144 por default, lo que pide ~36 GiB de KV
# cache — mucho más de lo que sobra en la GPU de Kaggle (P100/T4) después de cargar los pesos.
# Lo recortamos a un valor chico que alcanza de sobra para diálogo de juego.
MAX_MODEL_LEN = 16384

# Las GPUs gratis de Kaggle (P100/T4) tienen compute capability < 8, así que FLASH_ATTN (FA2)
# no es soportado, y el backend FLASHINFER por default falla al compilar su kernel JIT con
# ninja en esta imagen (falta libnvrtc.so.13). TORCH_SDPA es la implementación nativa de
# PyTorch: no compila nada al vuelo, así que evita ese problema (más lenta, pero confiable).
# La variable de entorno VLLM_ATTENTION_BACKEND está deprecada en esta versión de vLLM;
# ahora se fuerza con el flag --attention-config.backend.
cmd = [
    "vllm", "serve", "Qwen/Qwen3-4B-Instruct-2507",
    "--enable-lora",
    "--lora-modules", f"{LORA_ALIAS}=fireball-nlp/fireball-qwen3-4b-lora-10k",
    "--max-lora-rank", str(MAX_LORA_RANK),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--port", str(PORT),
    "--attention-config.backend", "TORCH_SDPA",
    # vLLM ya permite "*" por default, pero lo dejamos explícito.
    # IMPORTANTE: el valor debe ser un string JSON (comillas simples afuera, dobles adentro),
    # NO un string pelado como "*".
    "--allowed-origins", '["*"]',
]
if VLLM_API_KEY:
    cmd += ["--api-key", VLLM_API_KEY]

log_file = open("/kaggle/working/vllm.log", "w")
vllm_process = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)
print(f"vLLM arrancando (pid={vllm_process.pid}). Logs en /kaggle/working/vllm.log")

In [ ]:
# Celda 4 — esperar a que el servidor esté listo (carga del modelo base + LoRA puede tardar minutos)
import time
import requests

health_url = f"http://localhost:{PORT}/health"
for attempt in range(60):
    try:
        r = requests.get(health_url, timeout=2)
        if r.status_code == 200:
            print("vLLM está arriba.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    print("El servidor no levantó a tiempo. Últimas líneas del log:\n")
    with open("/kaggle/working/vllm.log") as f:
        print(f.read()[-3000:])

In [ ]:
# Celda 5 — abrir el túnel de ngrok e imprimir la URL pública
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)

if NGROK_DOMAIN:
    public_url = ngrok.connect(PORT, domain=NGROK_DOMAIN).public_url
else:
    public_url = ngrok.connect(PORT).public_url

assert public_url.startswith("https://"), "La URL debe ser https:// (GitHub Pages no permite mixed content con http://)"

print("=" * 60)
print("PEGÁ ESTO EN EL PANEL ⚙ LINK DE LA APP:\n")
print(f"  Base URL: {public_url}")
if VLLM_API_KEY:
    print(f"  API Key:  {VLLM_API_KEY}")
else:
    print("  API Key:  (dejar vacío, no configuraste VLLM_API_KEY)")
print("=" * 60)

### (Opcional) Verificación rápida antes de ir a la app

Para confirmar que el alias del LoRA quedó registrado correctamente, podés correr (en otra celda):
```python
!curl -s {public_url}/v1/models
```
y revisar que `fireball-qwen3-4b-lora-10k` aparezca en la lista.

In [ ]:
# Celda final — keep-alive
# Mantén esta celda corriendo mientras quieras que el servidor sea alcanzable.
# Cerrar la pestaña, que se cumpla el timeout de inactividad de Kaggle, o frenar la sesión
# manualmente, tira el túnel de ngrok y mata el proceso de vLLM. No hay forma de dejarlo
# corriendo "para siempre" en el tier gratuito de Kaggle: hay que prenderlo cada vez que se
# quiera jugar.
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Interrumpido. El túnel/proceso puede seguir vivo hasta que pares la sesión de Kaggle.")